# Project 1 — Retail Sales Performance Analysis
**Level:** Beginner  |  **Intern:** Sohan Nandi (CSN-OC-2026-474)  |  **Domain:** Data Analysis  |  **October 2026**

## Objective & Business Questions
Analyze retail sales and profit from the Superstore dataset.

- Which regions, states, categories, sub-categories generate the most sales and profit?
- Which products/segments/regions have low or negative profit?
- How do sales and profit vary over time?
- How do discounts relate to profitability?
- Shipping and order trends.

In [1]:
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def resolve_repo_root():
    cwd = Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").is_dir() and (parent / "projects").is_dir():
            return parent
        if (parent / "Dataset").is_dir():
            return parent
    return Path(__file__).resolve().parents[3]

REPO_ROOT = resolve_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
def fig_path(name):
    return str(OUT_DIR / name)

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUT_DIR)

# --- Load source data ---
SRC_FILE = DATASET_DIR / "Sample - Superstore.csv"
print("Source file:", SRC_FILE)
assert SRC_FILE.exists(), f"Source file missing: {SRC_FILE}"
df = pd.read_csv(SRC_FILE, encoding="latin1", parse_dates=["Order Date", "Ship Date"])
print("Loaded shape:", df.shape)


Repo root: C:\Users\sohan\OneDrive\Desktop\CyberNova
Dataset dir: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset
Output dir: outputs
Source file: C:\Users\sohan\OneDrive\Desktop\CyberNova\Dataset\Sample - Superstore.csv
Loaded shape: (9994, 21)


In [2]:
print("dimensions:", df.shape)
print("columns:", list(df.columns))
print("dtypes:"); print(df.dtypes.to_string())
print("first 5 rows:"); print(df.head().to_string())
print("--- numeric stats ---")
print(df[["Sales", "Quantity", "Discount", "Profit"]].describe().to_string())

dimensions: (9994, 21)
columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
dtypes:
Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code               int64
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
first 5 rows:
   Row ID     

In [3]:
print("null counts:"); print(df.isnull().sum().to_string())
print("duplicate rows:", int(df.duplicated().sum()))
print("unique Order IDs:", df["Order ID"].nunique(), "| rows:", len(df))
print("order-date range:", df["Order Date"].min().date(), "->", df["Order Date"].max().date())
print("discount values:", sorted(df["Discount"].dropna().unique().tolist()))
print("negative profit rows:", int((df["Profit"] < 0).sum()))
print("category:", df["Category"].value_counts().to_dict())
print("segment:", df["Segment"].value_counts().to_dict())
print("ship mode:", df["Ship Mode"].value_counts().to_dict())

null counts:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
duplicate rows: 0
unique Order IDs: 5009 | rows: 9994
order-date range: 2014-01-03 -> 2017-12-30
discount values: [0.0, 0.1, 0.15, 0.2, 0.3, 0.32, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8]
negative profit rows: 1871
category: {'Office Supplies': 6026, 'Furniture': 2121, 'Technology': 1847}
segment: {'Consumer': 5191, 'Corporate': 3020, 'Home Office': 1783}
ship mode: {'Standard Class': 5968, 'Second Class': 1945, 'First Class': 1538, 'Same Day': 543}


In [4]:
clean = df.copy()
assert clean["Order ID"].notna().all()
assert clean["Sales"].notna().all() and clean["Profit"].notna().all()
clean["Profit Margin"] = clean["Profit"] / clean["Sales"].replace(0, np.nan)
clean["Order Month"] = clean["Order Date"].dt.to_period("M").dt.to_timestamp()
clean["Order Quarter"] = clean["Order Date"].dt.to_period("Q").dt.to_timestamp()
clean["Is Loss"] = clean["Profit"] < 0
print("cleaned rows:", len(clean), "| cols:", clean.shape[1])
print("any Sales<=0:", bool((clean["Sales"] <= 0).any()))
print("derived margin mean:", round(clean["Profit Margin"].mean(), 4))
clean.to_csv(OUT_DIR / "cleaned_retail_sales.csv", index=False)
print("saved -> outputs/cleaned_retail_sales.csv")

cleaned rows: 9994 | cols: 25
any Sales<=0: False
derived margin mean: 0.1203


saved -> outputs/cleaned_retail_sales.csv

In [5]:
agg_region = clean.groupby("Region")[["Sales","Profit"]].sum()
agg_category = clean.groupby("Category")[["Sales","Profit"]].sum()
agg_subcat = clean.groupby("Sub-Category")["Profit"].sum().sort_values()
agg_state_profit = clean.groupby("State")["Profit"].sum().sort_values()
agg_segment = clean.groupby("Segment")[["Sales","Profit"]].sum()
monthly = clean.groupby("Order Month")[["Sales","Profit"]].sum()
quarterly = clean.groupby("Order Quarter")[["Sales","Profit"]].sum()
print("REGION:\n", agg_region.round(2).to_string())
print("CATEGORY:\n", agg_category.round(2).to_string())
print("SUBCATEGORY bottom 10:\n", agg_subcat.head(10).round(2).to_string())
print("SUBCATEGORY top 10:\n", agg_subcat.tail(10).round(2).to_string())
print("STATE bottom profit:\n", agg_state_profit.head(6).round(2).to_string())
print("SEGMENT:\n", agg_segment.round(2).to_string())
print("FIRST/MONTH\n", monthly.head(2).round(2).to_string())
print("LAST/MONTH\n", monthly.tail(2).round(2).to_string())

REGION:


              Sales     Profit
Region                       
Central  501239.89   39706.36
East     678781.24   91522.78
South    391721.90   46749.43
West     725457.82  108418.45
CATEGORY:
                      Sales     Profit
Category                             
Furniture        741999.80   18451.27
Office Supplies  719047.03  122490.80
Technology       836154.03  145454.95
SUBCATEGORY bottom 10:
 Sub-Category
Tables        -17725.48
Bookcases      -3472.56
Supplies       -1189.10
Fasteners        949.52
Machines        3384.76
Labels          5546.25
Art             6527.79
Envelopes       6964.18
Furnishings    13059.14
Appliances     18138.01
SUBCATEGORY top 10:
 Sub-Category
Envelopes       6964.18
Furnishings    13059.14
Appliances     18138.01
Storage        21278.83
Chairs         26590.17
Binders        30221.76
Paper          34053.57
Accessories    41936.64
Phones         44515.73
Copiers        55617.82
STATE bottom profit:
 State
Texas            -25729.36
Ohio        

In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
agg_region["Profit"].sort_values().plot(kind="barh", ax=axes[0,0], color="#1f77b4")
axes[0,0].set_title("Profit by Region"); axes[0,0].set_xlabel("Profit ($)")
agg_category.plot(kind="bar", ax=axes[0,1], color=["#2ca02c","#d62728","#9467bd"])
axes[0,1].set_title("Sales & Profit by Category"); axes[0,1].tick_params(axis="x", rotation=25); axes[0,1].legend()
agg_subcat.sort_values().head(8).plot(kind="barh", ax=axes[0,2], color="#d62728")
axes[0,2].set_title("Lowest-Profit Sub-Categories"); axes[0,2].set_xlabel("Profit ($)")
monthly["Sales"].plot(kind="line", ax=axes[1,0], marker="o", color="#ff7f0e")
axes[1,0].set_title("Monthly Sales Trend"); axes[1,0].set_xlabel("Month"); axes[1,0].tick_params(axis="x", rotation=45)
sample = clean.sample(n=1200, random_state=1)
axes[1,1].scatter(sample["Discount"], sample["Profit"], alpha=0.5, s=12)
axes[1,1].set_title("Discount vs Profit (sample)"); axes[1,1].set_xlabel("Discount"); axes[1,1].set_ylabel("Profit ($)")
clean["Ship Mode"].value_counts().plot(kind="bar", ax=axes[1,2], color="#7f7f7f")
axes[1,2].set_title("Orders by Ship Mode"); axes[1,2].tick_params(axis="x", rotation=25); axes[1,2].set_ylabel("Count")
fig.tight_layout(); fig.savefig(fig_path("charts_retail.png"), dpi=150, bbox_inches="tight"); plt.close(fig)
print("saved outputs/charts_retail.png")

saved outputs/charts_retail.png


In [7]:
print("KEY FINDINGS (retail):")
print("1. Sales $%.2f | Profit $%.2f | margin %.1f%% | orders %d | qty %d" % (
    clean["Sales"].sum(), clean["Profit"].sum(), clean["Profit"].sum()/clean["Sales"].sum()*100,
    clean["Order ID"].nunique(), clean["Quantity"].sum()))
print("2. West profit $%.2f; Central $%.2f (least)" % (
    agg_region.loc["West","Profit"], agg_region.loc["Central","Profit"]))
print("3. Furniture profit $%.2f on $%.2f sales (lowest margin category); Office Supplies $%.2f" % (
    agg_category.loc["Furniture","Profit"], agg_category.loc["Furniture","Sales"], agg_category.loc["Office Supplies","Profit"]))
print("4. Tables profit $%.2f (biggest loss); Texas state profit $%.2f (lossiest)" % (
    agg_subcat.loc["Tables"], agg_state_profit.loc["Texas"]))
print("5. %d of %d orders (%.1f%%) lose money, totalling $%.2f" % (
    int(clean["Is Loss"].sum()), len(clean), clean["Is Loss"].mean()*100,
    clean.loc[clean["Is Loss"],"Profit"].sum()))
print("6. Discount-profit corr: %.2f" % clean["Discount"].corr(clean["Profit"]))
print("RECOMMENDATIONS:")
print("- Review >20%% discount loss-making orders, esp. Tables/Bookcases; negotiate supplier costs.")
print("- Investigate Texas/Ohio/Pennsylvania pricing/logistics.")
print("- Improve Furniture/Table margin toward Office Supplies levels.")

KEY FINDINGS (retail):
1. Sales $2297200.86 | Profit $286397.02 | margin 12.5% | orders 5009 | qty 37873
2. West profit $108418.45; Central $39706.36 (least)
3. Furniture profit $18451.27 on $741999.80 sales (lowest margin category); Office Supplies $122490.80
4. Tables profit $-17725.48 (biggest loss); Texas state profit $-25729.36 (lossiest)
5. 1871 of 9994 orders (18.7%) lose money, totalling $-156131.29
6. Discount-profit corr: -0.22
RECOMMENDATIONS:
- Review >20%% discount loss-making orders, esp. Tables/Bookcases; negotiate supplier costs.
- Investigate Texas/Ohio/Pennsylvania pricing/logistics.
- Improve Furniture/Table margin toward Office Supplies levels.


In [8]:
print("LIMITATIONS:")
print("- Observational: discount-profit is association, not causation.")
print("- Superstore is a sample dataset; may not generalize to a real retailer.")
print("- Order ID != line item; counts here are order-line counts (one order can span rows).")
print("- Single file; no customer-table enrichment.")
print("CONCLUSION: Profitable overall (~12.5%% margin) but driven by a few categories while a meaningful minority of order-lines lose money. Discount + category review is the clearest lever.")

LIMITATIONS:
- Observational: discount-profit is association, not causation.
- Superstore is a sample dataset; may not generalize to a real retailer.
- Order ID != line item; counts here are order-line counts (one order can span rows).
- Single file; no customer-table enrichment.
CONCLUSION: Profitable overall (~12.5%% margin) but driven by a few categories while a meaningful minority of order-lines lose money. Discount + category review is the clearest lever.
